In [1]:
import os
import langchain
import requests
import sys
import re
from langchain.document_loaders.generic import GenericLoader

#from langchain.chat_models import ChatOpenAI
#from langchain.llms import OpenAI
from langchain_community.llms import Ollama

from langchain.document_loaders.parsers import LanguageParser
#splits the python script into functions
from langchain.text_splitter import Language
from langchain.prompts import PromptTemplate, ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.messages import SystemMessage

#create chunks for documents
from langchain.text_splitter import RecursiveCharacterTextSplitter
#create ebeddings
from langchain.embeddings import HuggingFaceBgeEmbeddings
#create chain
from langchain.chains import RetrievalQA
#stores embeddings to faiss vectorstore
from langchain.vectorstores import FAISS
from langchain_community.llms import Ollama

# Creating Vector DB

In [ ]:

'''
# Specify the path to your local folder containing .py files
path = "./ollamawork/Finetuning_2025/apitests"

# Function to load files from the local folder
def load_local_docs(path):
    loader = GenericLoader.from_filesystem(
        path=path,
        glob="*",  # This matches all files in the folder
        suffixes=[".py"],  # Load only Python files
        parser=LanguageParser()  # Use LanguageParser to process the files
    )
    return loader.load()

# Load documents from the specified path
docs = load_local_docs(path)

# Remove unwanted empty chunks and unnecessary code lines
docs_without_empty_chunk = [
    i for i in docs if not (str(i.page_content).startswith("# Code for:")
                            or str(i.page_content).startswith("import")
                            or str(i.page_content).startswith("from"))
]

# Create text chunks
python_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON, chunk_size=2500, chunk_overlap=200
)
texts = python_splitter.split_documents(docs_without_empty_chunk)

# Define the embedding model (bge-base-en)
model_name = "BAAI/bge-base-en"
encode_kwargs = {'normalize_embeddings': True}
model_norm = HuggingFaceBgeEmbeddings(
    model_name=model_name,
    model_kwargs={'device': 'cuda'},  # Use GPU for faster processing
    encode_kwargs=encode_kwargs
)

# Create FAISS vector database
vectordb = FAISS.from_documents(documents=texts, embedding=model_norm)

# Path to save the FAISS index
faiss_path = "./ollamawork/Finetuning_2025/faiss_colama"

# Ensure the directory exists before saving
os.makedirs(faiss_path, exist_ok=True)

# Save the FAISS database
vectordb.save_local(faiss_path)
print(f"FAISS index saved to {faiss_path}")'''


In [ ]:
import torch
import os
from langchain_core.prompts import PromptTemplate
from langchain_community.vectorstores import FAISS
from langchain.chains import RetrievalQA
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from langchain_community.llms import HuggingFacePipeline

def get_between_triple_quotes(string):
    triple_quote = '"""'
    start = string.find(triple_quote)  # Find the first triple quote
    end = string.find(triple_quote, start + 3)  # Find the next triple quote after the first
    if start != -1 and end != -1 and end > start:
        return string[start + 3:end].strip()  # Extract the content between the triple quotes
    else:
        return ""  # Return an empty string if triple quotes are not found

    
    
def filt_s(s):
    if s:  # Check if the string is not empty
        substrings = s.split('\n')  # Split the string by newlines
        if len(substrings) > 1:  # Check if there is more than one line
            desired_substring = substrings[1]  # Get the second line
            cleaned_substring = desired_substring.strip()  # Strip leading/trailing whitespace
            return cleaned_substring
        else:
            return s.strip()  # Return the string itself if it's a single line
    else:
        return '' 


# Load the fine-tuned model
model_name = "./ollamawork/Finetuning_2025/Colama_Rag+Finetune/Result_phi2_3"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="cuda").to("cuda")

# Create a text generation pipeline


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
tokenizer.pad_token=tokenizer.eos_token
tokenizer.truncation_side="left"
torch.manual_seed(42)
text_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_length=2048,
    temperature=0.2,
    #top_p=0.2,
    #top_k=10,
    #repetition_penalty=1,
   # device=0  # Use GPU
     do_sample=True,
     #seed=42
     truncation=True
)

# Wrap the pipeline with HuggingFacePipeline for LangChain compatibility
chat_model = HuggingFacePipeline(pipeline=text_pipeline)

# Load the FAISS vector store
faiss_path = "./ollamawork/Finetuning_2025/faiss_colama"

def get_embedding_model():
    """Returns the embedding model using HuggingFaceBgeEmbeddings with GPU support."""
    model_name = "BAAI/bge-base-en"
    encode_kwargs = {"normalize_embeddings": True}
    model_norm = HuggingFaceBgeEmbeddings(
        model_name=model_name,
        model_kwargs={"device": "cuda"},
        encode_kwargs=encode_kwargs
    )
    return model_norm

def load_vectordb(save_path: str):
    """Loads the vector database from the specified path."""
    model_norm = get_embedding_model()
    vectordb = FAISS.load_local(save_path, model_norm, allow_dangerous_deserialization=True)
    return vectordb

vectordb = load_vectordb(faiss_path)


# Define the prompt template
prompt_template = """
###System:
You are an automation engineer, good at writing test code.
Generate comprehensive test code for the user query, drawing upon the knowledge and functionality provided within a given context. 
Understand the context and generate code for the API specifications mentioned in the query. 
The provided context is the test code for an already existing API specification or it may be the content of some library file. 
Query is the specific REST API for which a test function is to be written.

###Follow the below guidelines:
1. Do not return results that are not in the context.
2. The library files are already written; call the library functions where needed instead of rewriting them.
3. Use docstrings to specify the function's functionality.
4. Use appropriate function names.
5. Assert correct status code and message as given in the query.
6. Use decorators where necessary.
Think step by step and use the same coding style as that of the context.

## Context:
{context}

Write down a python function that tests the API scenario described below. Make sure you use appropriate docstring while writing code. Below are the endpoint, inputs, and output of the function as well.
{question}

"""

PROMPT = PromptTemplate(template=prompt_template, input_variables=["context", "question"])

# Define RetrievalQA with the fine-tuned model
qa = RetrievalQA.from_chain_type(
    llm=chat_model,  # ✅ FIXED: Using the wrapped model
    chain_type="stuff",
    chain_type_kwargs={"prompt": PROMPT},
    retriever=vectordb.as_retriever(search_kwargs={'k': 3})
)
import pandas as pd
val = pd.read_excel("./ollamawork/Finetuning_2025/Finetune_colama-llama3.1/train_750_library_files.xlsx",sheet_name='eval')


with open("generated_test_code_phi2_2_0.2.txt", "w", encoding="utf-8") as file:
    for i in range(len(val)):
        print(str(i+1) + " is started")
        desc = filt_s(get_between_triple_quotes(val['Test_Code'][i]))
        # Define the query
        question = f"""
        ###Endpoint:
        {val['Endpoint'][i]}\n
        ###Description:
        {val['Description'][i]}\n
        {desc}\n
        ###Inputs:
        {val['Inputs'][i]}\n
        ###Output:
        {val['Output'][i]}
        ###Answer:\n
        
        """

        # Retrieve relevant test cases
        retrieved_docs = qa.retriever.get_relevant_documents(question)

        # Generate the test case code
        result = qa({"query": question})

        # Extract result content
        generated_code = result["result"]

        # Find the index of "## Query ##"
        query_index = generated_code.find("###Endpoint:")

        if query_index != -1:
            gen = generated_code[query_index + len("###Endpoint:"):].strip()
        else:
            gen = generated_code  # If "## Query ##" is not found, store the full result

        # Write to the file
        file.write(f"Test Case {i+1}:\n")
        file.write(gen + "\n\n" + "-" * 100 + "\n\n")

print("Generated test cases saved to 'generated_test_code_phi2.txt'")


1 is started
2 is started
3 is started
4 is started
5 is started
6 is started
7 is started
8 is started
9 is started
10 is started
11 is started
12 is started
13 is started
14 is started
15 is started
16 is started
17 is started
18 is started
19 is started
20 is started
21 is started
22 is started
23 is started
24 is started
25 is started
26 is started
Generated test cases saved to 'generated_test_code_phi2.txt'


In [ ]:
tokenizer.pad_token=tokenizer.eos_token
tokenizer.truncation_side="left"
torch.manual_seed(42)
text_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_length=2048,
    temperature=0.2,
    #top_p=0.2,
    #top_k=10,
    #repetition_penalty=1,
   # device=0  # Use GPU
     do_sample=True,
     #seed=42
     truncation=True
)

# Wrap the pipeline with HuggingFacePipeline for LangChain compatibility
chat_model = HuggingFacePipeline(pipeline=text_pipeline)

# Load the FAISS vector store
faiss_path = "./ollamawork/Finetuning_2025/faiss_colama"

def get_embedding_model():
    """Returns the embedding model using HuggingFaceBgeEmbeddings with GPU support."""
    model_name = "BAAI/bge-base-en"
    encode_kwargs = {"normalize_embeddings": True}
    model_norm = HuggingFaceBgeEmbeddings(
        model_name=model_name,
        model_kwargs={"device": "cuda"},
        encode_kwargs=encode_kwargs
    )
    return model_norm

def load_vectordb(save_path: str):
    """Loads the vector database from the specified path."""
    model_norm = get_embedding_model()
    vectordb = FAISS.load_local(save_path, model_norm, allow_dangerous_deserialization=True)
    return vectordb

vectordb = load_vectordb(faiss_path)


# Define the prompt template
prompt_template = """
###System:
You are an automation engineer, good at writing test code.
Generate comprehensive test code for the user query, drawing upon the knowledge and functionality provided within a given context. 
Understand the context and generate code for the API specifications mentioned in the query. 
The provided context is the test code for an already existing API specification or it may be the content of some library file. 
Query is the specific REST API for which a test function is to be written.

###Follow the below guidelines:
1. Do not return results that are not in the context.
2. The library files are already written; call the library functions where needed instead of rewriting them.
3. Use docstrings to specify the function's functionality.
4. Use appropriate function names.
5. Assert correct status code and message as given in the query.
6. Use decorators where necessary.
Think step by step and use the same coding style as that of the context.

## Context:
{context}

Write down a python function that tests the API scenario described below. Make sure you use appropriate docstring while writing code. Below are the endpoint, inputs, and output of the function as well.
{question}

"""

PROMPT = PromptTemplate(template=prompt_template, input_variables=["context", "question"])

# Define RetrievalQA with the fine-tuned model
qa = RetrievalQA.from_chain_type(
    llm=chat_model,  # ✅ FIXED: Using the wrapped model
    chain_type="stuff",
    chain_type_kwargs={"prompt": PROMPT},
    retriever=vectordb.as_retriever(search_kwargs={'k': 3})
)
import pandas as pd
val = pd.read_excel("./ollamawork/Finetuning_2025/Finetune_colama-llama3.1/train_750_library_files.xlsx",sheet_name='eval')



question = f"""
###Endpoint: /ilibrary/rest/details/{UUID}/

###Description:
Fetching details of public island library from private island.

To check machine type with public island.

###Inputs:
machine1 = {{
  "uuid": r1.json()["uuid"],
  "nics": {{
    "add": [
      {{
        "mac": "auto",
        "type": "bridge",
        "model": networks[0].get("model", "virtio"),
        "segment": "Default Public Segment"
      }}
    ]
  }}
}}

params = {{
  "name": "Machine1",
  "is_public": False,
  "machines": {{
    "add": [machine1]
  }}
}}

###Output:
{{
  "response": "success"
}}
"""

# Retrieve relevant test cases
retrieved_docs = qa.retriever.get_relevant_documents(question)

# Generate the test case code
result = qa({"query": question})

# Extract result content
generated_code = result["result"]



print(generated_code)
